# SkyOps Week 3 — Data Exploration Notebook

**Project:** SkyOps Airline Delay Command Center  
**Project ID:** P07  
**Project short name:** skyops  
**Databricks volume:** `/Volumes/p07-skyops/default/skyops`

This notebook explores the Week-3 source files only. It loads the four source CSVs, profiles them with Spark SQL, checks quality signals, and creates one small Bronze demo table plus one lineage demo view.

## Week-3 mission

Week 3 is about understanding the raw sources well enough to defend the data model later. The goal is to inspect grain, keys, counts, distributions, missing values, obvious defects, and parent-child relationships before any full Bronze/Silver/Gold build.

## Week-3 versus Week-4 boundary

This notebook stays in exploration scope. It does **not** build the full Bronze layer, Silver cleaning, Gold metrics, or any streaming pipeline.

## Databricks language choice

Spark SQL is the primary exploration language in this notebook. PySpark is used only for simple file reads, DataFrame creation, display, schema inspection, and lightweight counts.

In [ ]:
from pyspark.sql import functions as F

volume_path = "/Volumes/p07-skyops/default/skyops"
source_files = {
    "flights": f"{volume_path}/flights.csv",
    "airports": f"{volume_path}/airports.csv",
    "carriers": f"{volume_path}/carriers.csv",
    "routes": f"{volume_path}/routes.csv",
}

print("Source files discovered:")
for name, path in source_files.items():
    print(f"- {name}: {path}")

## Source file inspection

The first check is simply to confirm the files exist in the expected Databricks Volume path. This is the path authority for the notebook.

In [ ]:
# Read the four source files as Spark DataFrames
flights_df = spark.read.option("header", True).option("inferSchema", True).csv(source_files["flights"])
airports_df = spark.read.option("header", True).option("inferSchema", True).csv(source_files["airports"])
carriers_df = spark.read.option("header", True).option("inferSchema", True).csv(source_files["carriers"])
routes_df = spark.read.option("header", True).option("inferSchema", True).csv(source_files["routes"])

print("DataFrames created: flights_df, airports_df, carriers_df, routes_df")

## Flights DataFrame

The notebook creates one DataFrame for each source file and displays it. This helps verify the raw shape before any joins or cleaning.

In [ ]:
print("Schema for flights_df")
flights_df.printSchema()
display(flights_df)

In [ ]:
flights_df.createOrReplaceTempView("flights")
print("Temporary SQL view created: flights")

## Airports DataFrame

The notebook creates one DataFrame for each source file and displays it. This helps verify the raw shape before any joins or cleaning.

In [ ]:
print("Schema for airports_df")
airports_df.printSchema()
display(airports_df)

In [ ]:
airports_df.createOrReplaceTempView("airports")
print("Temporary SQL view created: airports")

## Carriers DataFrame

The notebook creates one DataFrame for each source file and displays it. This helps verify the raw shape before any joins or cleaning.

In [ ]:
print("Schema for carriers_df")
carriers_df.printSchema()
display(carriers_df)

In [ ]:
carriers_df.createOrReplaceTempView("carriers")
print("Temporary SQL view created: carriers")

## Routes DataFrame

The notebook creates one DataFrame for each source file and displays it. This helps verify the raw shape before any joins or cleaning.

In [ ]:
print("Schema for routes_df")
routes_df.printSchema()
display(routes_df)

In [ ]:
routes_df.createOrReplaceTempView("routes")
print("Temporary SQL view created: routes")

## Grain and business-key notes

The source files suggest these working grains:

- **flights**: one row per flight leg / flight record
- **airports**: one row per airport
- **carriers**: one row per carrier
- **routes**: one row per origin-destination route

The flight record is the central fact-like source. Airports and carriers act as reference-style master data, while routes provide route context for origin-destination pairs.

In [ ]:
spark.sql("""
SELECT 'flights' AS source, COUNT(*) AS physical_rows, COUNT(DISTINCT source_record_key) AS distinct_business_keys
FROM flights
UNION ALL
SELECT 'airports' AS source, COUNT(*) AS physical_rows, COUNT(DISTINCT airport_code) AS distinct_business_keys
FROM airports
UNION ALL
SELECT 'carriers' AS source, COUNT(*) AS physical_rows, COUNT(DISTINCT carrier_code) AS distinct_business_keys
FROM carriers
UNION ALL
SELECT 'routes' AS source, COUNT(*) AS physical_rows, COUNT(DISTINCT route_id) AS distinct_business_keys
FROM routes
""").show(truncate=False)

The comparison between physical rows and distinct business keys shows whether duplicates or repeated business entities exist. In a clean master file, these numbers should usually match. For the flight source, a row count and business-key count mismatch would be an important defect signal.

In [ ]:
spark.sql("""
SELECT reporting_carrier, COUNT(*) AS flight_rows
FROM flights
GROUP BY reporting_carrier
ORDER BY flight_rows DESC
LIMIT 10
""").show(truncate=False)

spark.sql("""
SELECT origin_airport_code, COUNT(*) AS departures
FROM flights
GROUP BY origin_airport_code
ORDER BY departures DESC
LIMIT 10
""").show(truncate=False)

spark.sql("""
SELECT destination_airport_code, COUNT(*) AS arrivals
FROM flights
GROUP BY destination_airport_code
ORDER BY arrivals DESC
LIMIT 10
""").show(truncate=False)

## Date and numeric ranges

Flight dates, delay minutes, and distance values are the main fields worth checking first. The notebook records the query results without inventing any counts or ranges.

In [ ]:
spark.sql("""
SELECT 
  MIN(flight_date) AS min_flight_date,
  MAX(flight_date) AS max_flight_date
FROM flights
""").show(truncate=False)

spark.sql("""
SELECT
  MIN(distance_miles) AS min_distance_miles,
  MAX(distance_miles) AS max_distance_miles,
  MIN(arrival_delay_minutes) AS min_arrival_delay_minutes,
  MAX(arrival_delay_minutes) AS max_arrival_delay_minutes,
  MIN(departure_delay_minutes) AS min_departure_delay_minutes,
  MAX(departure_delay_minutes) AS max_departure_delay_minutes
FROM flights
""").show(truncate=False)

## Missing-value checks

The next queries look for nulls in fields that matter for joins and operational interpretation.

In [ ]:
spark.sql("""
SELECT
  SUM(CASE WHEN reporting_carrier IS NULL THEN 1 ELSE 0 END) AS null_reporting_carrier,
  SUM(CASE WHEN origin_airport_code IS NULL THEN 1 ELSE 0 END) AS null_origin_airport_code,
  SUM(CASE WHEN destination_airport_code IS NULL THEN 1 ELSE 0 END) AS null_destination_airport_code,
  SUM(CASE WHEN flight_date IS NULL THEN 1 ELSE 0 END) AS null_flight_date,
  SUM(CASE WHEN source_record_key IS NULL THEN 1 ELSE 0 END) AS null_source_record_key
FROM flights
""").show(truncate=False)

spark.sql("""
SELECT
  SUM(CASE WHEN airport_code IS NULL THEN 1 ELSE 0 END) AS null_airport_code,
  SUM(CASE WHEN airport_name IS NULL THEN 1 ELSE 0 END) AS null_airport_name
FROM airports
""").show(truncate=False)

spark.sql("""
SELECT
  SUM(CASE WHEN carrier_code IS NULL THEN 1 ELSE 0 END) AS null_carrier_code,
  SUM(CASE WHEN carrier_name IS NULL THEN 1 ELSE 0 END) AS null_carrier_name
FROM carriers
""").show(truncate=False)

## Negative or impossible-value checks

For the flight source, negative delay minutes and contradictory cancellation fields are the kinds of issues that can break trust in the numbers.

In [ ]:
spark.sql("""
SELECT
  SUM(CASE WHEN arrival_delay_minutes < 0 THEN 1 ELSE 0 END) AS negative_arrival_delay_rows,
  SUM(CASE WHEN departure_delay_minutes < 0 THEN 1 ELSE 0 END) AS negative_departure_delay_rows,
  SUM(CASE WHEN cancelled_flag = 1 AND actual_arrival_hhmm IS NOT NULL THEN 1 ELSE 0 END) AS cancelled_with_arrival_time_rows,
  SUM(CASE WHEN cancelled_flag = 1 AND actual_departure_hhmm IS NOT NULL THEN 1 ELSE 0 END) AS cancelled_with_departure_time_rows
FROM flights
""").show(truncate=False)

## Relationship checks

The flight source should join cleanly to the reference sources. These anti-join checks help reveal invalid references.

In [ ]:
spark.sql("""
SELECT COUNT(*) AS invalid_origin_airport_references
FROM flights f
LEFT ANTI JOIN airports a
  ON f.origin_airport_code = a.airport_code
""").show()

spark.sql("""
SELECT COUNT(*) AS invalid_destination_airport_references
FROM flights f
LEFT ANTI JOIN airports a
  ON f.destination_airport_code = a.airport_code
""").show()

spark.sql("""
SELECT COUNT(*) AS invalid_carrier_references
FROM flights f
LEFT ANTI JOIN carriers c
  ON f.reporting_carrier = c.carrier_code
""").show()

spark.sql("""
SELECT COUNT(*) AS invalid_route_references
FROM routes r
LEFT ANTI JOIN airports a1
  ON r.origin_airport_code = a1.airport_code
LEFT ANTI JOIN airports a2
  ON r.destination_airport_code = a2.airport_code
""").show()

## One meaningful business question

A simple question for Week 3 is: which carriers show the highest average arrival delay in the extracted sample?

In [ ]:
spark.sql("""
SELECT
  reporting_carrier,
  COUNT(*) AS flight_rows,
  AVG(arrival_delay_minutes) AS avg_arrival_delay_minutes
FROM flights
GROUP BY reporting_carrier
ORDER BY avg_arrival_delay_minutes DESC
LIMIT 10
""").show(truncate=False)

## Bronze demo table

This notebook creates exactly one small Bronze demo table using the main transaction entity, `flights`. It is a demonstration object only, not the full Bronze layer.

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.default.skyops_week03_bronze_demo_flights
USING DELTA
AS
SELECT *
FROM flights
""")

spark.sql("""
DESCRIBE DETAIL workspace.default.skyops_week03_bronze_demo_flights
""").show(truncate=False)

spark.sql("""
DESCRIBE HISTORY workspace.default.skyops_week03_bronze_demo_flights
""").show(truncate=False)

### Simple source-to-demo row-count check

Run and record actual result.

In [ ]:
spark.sql("""
SELECT 
  (SELECT COUNT(*) FROM flights) AS source_row_count,
  (SELECT COUNT(*) FROM workspace.default.skyops_week03_bronze_demo_flights) AS bronze_demo_row_count
""").show(truncate=False)

## Lineage demo view

The lineage demo view is a simple downstream object that can be traced from the Bronze demo table.

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.default.skyops_week03_lineage_demo_view AS
SELECT
  reporting_carrier,
  origin_airport_code,
  destination_airport_code,
  flight_date,
  arrival_delay_minutes
FROM workspace.default.skyops_week03_bronze_demo_flights
""")

spark.sql("""
SELECT COUNT(*) AS lineage_view_rows
FROM workspace.default.skyops_week03_lineage_demo_view
""").show()

## Catalog Explorer lineage instructions

Open the Databricks Catalog Explorer, locate:

- `workspace.default.skyops_week03_bronze_demo_flights`
- `workspace.default.skyops_week03_lineage_demo_view`

Then inspect the lineage panel to confirm the downstream relationship from the demo table to the demo view.

## Evidence checklist

- Source files loaded from the exact Volume path
- One DataFrame created for each source
- One temporary SQL view created for each source
- Grain and key notes captured
- Row counts and distinct-key counts checked
- Missing values checked
- Negative / impossible values checked
- Relationship / anti-join checks included
- One business question answered
- Exactly one Bronze demo table created
- Exactly one lineage demo view created
- DESCRIBE DETAIL run
- DESCRIBE HISTORY run
- Week-3 versus Week-4 boundary maintained

## Intern defence questions

1. What is the grain of each source file?
2. Why is `flights` the central source?
3. Which columns act as business keys?
4. How do you know the reference joins are valid?
5. What defects would make the flight metrics untrustworthy?
6. Why is this notebook still Week 3 and not Week 4?

## Conversion validation report

| Item | Status |
|---|---|
| PageLoop references remaining | Run and verify in notebook text |
| Assigned source files covered | flights, airports, carriers, routes |
| DataFrames created | flights_df, airports_df, carriers_df, routes_df |
| SQL views created | flights, airports, carriers, routes |
| Relationship checks included | Yes |
| Bronze demo-table count | 1 |
| Lineage-view count | 1 |
| Week-4 overlap check | No full Bronze/Silver/Gold/streaming build included |
| Notebook structural validation result | Complete exploration notebook structure provided |